In [ ]:
import csv
import time
from scrapling.fetchers import StealthyFetcher

# Global/top-level browser instance so the session stays alive across cells.
# Do NOT wrap this in a context manager.
fetcher = StealthyFetcher(headless=False)

# Shared state containers used by later cells
login_url = "https://www.semrush.com/login/"
scraped_rows = []
page = None


In [ ]:
# Navigate to the login page in the visible browser.
# Top-level await is intentionally used for notebook compatibility.
try:
    page = await fetcher.get(login_url)
except AttributeError:
    # Fallback for Scrapling variants that expose browser navigation via fetch().
    page = await fetcher.fetch(login_url)

input("Please login manually in the visual browser, solve any Captcha, and press ENTER here to continue...")


In [ ]:
time.sleep(5)

scraped_rows = []
current_url = login_url

# Try to reuse the authenticated browser/session state after manual login.
try:
    current_url = page.url
except Exception:
    try:
        current_url = fetcher.page.url
    except Exception:
        current_url = login_url

try:
    refreshed_page = await fetcher.get(current_url)
except AttributeError:
    refreshed_page = await fetcher.fetch(current_url)
except Exception as navigation_error:
    print(f"Could not refresh the authenticated page: {navigation_error}")
    refreshed_page = page

# Strategy: prefer semantic table selectors and text/structure-based parsing.
table_candidates = []

try:
    table_candidates = refreshed_page.xpath("//table[.//th or .//td]")
except Exception as table_xpath_error:
    print(f"Primary table lookup failed: {table_xpath_error}")

if not table_candidates:
    try:
        table_candidates = refreshed_page.css("table")
    except Exception as table_css_error:
        print(f"Fallback table lookup failed: {table_css_error}")

target_table = table_candidates[0] if table_candidates else None

target_headers = ["Keyword", "Position", "Volume", "KD"]
headers = []
try:
    if target_table is not None:
        headers = [
            header.strip()
            for header in target_table.xpath(".//thead//th//text()").getall()
            if header and header.strip()
        ]
except Exception as header_error:
    print(f"Header extraction failed: {header_error}")

if not headers:
    headers = target_headers.copy()

header_lookup = {header.lower(): index for index, header in enumerate(headers)}

def pick_value(cell_texts, preferred_names, fallback_index):
    for preferred_name in preferred_names:
        matched_index = header_lookup.get(preferred_name.lower())
        if matched_index is not None and matched_index < len(cell_texts):
            return cell_texts[matched_index]
    return cell_texts[fallback_index] if fallback_index < len(cell_texts) else ""

row_nodes = []
try:
    if target_table is not None:
        row_nodes = target_table.xpath(".//tbody/tr")
except Exception as tbody_error:
    print(f"tbody row extraction failed: {tbody_error}")

if not row_nodes:
    try:
        if target_table is not None:
            row_nodes = target_table.xpath(".//tr[position() > 1]")
    except Exception as generic_row_error:
        print(f"Generic row extraction failed: {generic_row_error}")

for row_index, row_node in enumerate(row_nodes, start=1):
    try:
        cell_texts = [
            cell.strip()
            for cell in row_node.xpath("./th//text() | ./td//text()").getall()
            if cell and cell.strip()
        ]

        if not cell_texts:
            continue

        normalized_row = {
            "Keyword": pick_value(cell_texts, ["keyword", "keywords"], 0),
            "Position": pick_value(cell_texts, ["position", "pos"], 1),
            "Volume": pick_value(cell_texts, ["volume"], 2),
            "KD": pick_value(cell_texts, ["kd", "keyword difficulty"], 3),
        }

        scraped_rows.append(normalized_row)
    except Exception as row_error:
        print(f"Skipping row {row_index} because of parsing error: {row_error}")

if not scraped_rows:
    print("No table rows were extracted. Verify the target page and adjust the semantic selectors if needed.")
else:
    print(f"Collected {len(scraped_rows)} rows.")
    scraped_rows[:3]


In [ ]:
output_file = "hasil_scraping.csv"

if scraped_rows:
    fieldnames = list(scraped_rows[0].keys())
    with open(output_file, "w", encoding="utf-8", newline="") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
        writer.writeheader()
        for row in scraped_rows:
            safe_row = {key: ("" if value is None else value) for key, value in row.items()}
            writer.writerow(safe_row)

    print(f"Saved {len(scraped_rows)} rows to {output_file}")
else:
    print("No data was exported because scraped_rows is empty.")

fetcher.close()
print("Browser session closed.")
